# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hicham1236/Intern-Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** Target pages that are stale (older than 180 days) AND have significant visibility (greater than 500 impressions in the last 90 days). The score is the total number of impressions, meaning older, high-traffic pages rank at the very top.
Reason Codes:

**STALE_HIGH_VOL:** Flagged for review (Meets age and volume thresholds).

**OK:** No action needed (Page is fresh or low volume).

In [2]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Setup environment and load data
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df['impressions_90d'] > 0].copy()
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Data loaded: {len(df)} active pages ready for rule processing.")

Data loaded: 30000 active pages ready for rule processing.


## 2. Build the ranked queue (writes the CSV)

We will encode the rule using simple boolean logic. The baseline_score will be 0 for pages that don't meet the rule, and equal to their impressions_90d if they do. We will then sort descending to build the queue and export it to work/outputs/.

In [3]:
#encode the rule and score
is_stale = df['content_age_days'] >= 180
is_visible = df['impressions_90d'] >= 500

df['baseline_score'] = (is_stale & is_visible).astype(int) * df['impressions_90d']
df['action_label'] = np.where(df['baseline_score'] > 0, "REVIEW_FOR_REFRESH", "NO_ACTION")
df['reason_code'] = np.where(df['baseline_score'] > 0, "STALE_HIGH_VOL", "OK")

# Build the queue (sort by score descending)
queue = df.sort_values('baseline_score', ascending=False).copy()

# Write the CSV
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
queue.to_csv(csv_path, index=False)
print(f"✅ Ranked queue successfully written to: {csv_path}")

✅ Ranked queue successfully written to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

**Top-20 Review Analysis:**

**Action & Reason Code:** All top 20 pages share the REVIEW_FOR_REFRESH action and STALE_HIGH_VOL reason code.

**Confidence Note:** Confidence is high that these pages are valuable (due to high impressions), but confidence that they are actually decaying is mixed.

What would make it wrong: This simple rule assumes age + volume = decay. It will be completely wrong if a page is a highly successful evergreen guide (e.g., "How to boil an egg"). Re-writing those pages would waste editorial time and risk losing current rankings.

In [4]:
#Display and analyze the Top 20
top_20 = queue.head(20)

display_cols = ['content_age_days', 'impressions_90d', 'baseline_score', 'reason_code', 'is_declining']
print("--- Top 20 Pages in the Queue ---")
display(top_20[display_cols])

# Check how accurate our rule actually is on the top 20
precision_at_20 = top_20['is_declining'].mean()
print(f"\nRule Precision@20: {precision_at_20:.2f} (Meaning {int(precision_at_20*100)}% of our top 20 are actually decaying)")

--- Top 20 Pages in the Queue ---


,content_age_days,impressions_90d,baseline_score,reason_code,is_declining
6653,537,517715,517715,STALE_HIGH_VOL,1
17812,445,517109,517109,STALE_HIGH_VOL,0
26844,445,509252,509252,STALE_HIGH_VOL,1
21819,445,463103,463103,STALE_HIGH_VOL,1
29400,299,443434,443434,STALE_HIGH_VOL,0
29879,482,416180,416180,STALE_HIGH_VOL,1
13537,362,347399,347399,STALE_HIGH_VOL,1
18870,445,345111,345111,STALE_HIGH_VOL,0
21565,445,309192,309192,STALE_HIGH_VOL,1
16811,224,288426,288426,STALE_HIGH_VOL,0



Rule Precision@20: 0.50 (Meaning 50% of our top 20 are actually decaying)


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.